# This notebook shows the steps to prepare the input files for Chapter 2

In [ ]:
import pandas as pd

## Extract the coordinates of the 5'leader and 3'UTR regions 

In [ ]:
# These files were downloaded from https://github.com/saketkc/riboraptor/tree/master/riboraptor/annotation/hg38
utr5 = pd.read_csv('../ref/hg38/utr5_hg38.bed.gz', sep='\t', header=None)
utr3 = pd.read_csv('../ref/hg38/utr3_hg38.bed.gz', sep='\t', header=None)
utr5[3] = 'pc.UTR5'
utr3[3] = 'pc.UTR3'
utr5.to_csv('../ref/hg38/utr5_hg38.bed',sep="\t", index=None, header=None)
utr3.to_csv('../ref/hg38/utr3_hg38.bed',sep="\t", index=None, header=None)
! gzip ../ref/hg38/utr5_hg38.bed
! gzip ../ref/hg38/utr3_hg38.bed

# Merge and sort both files
! cat \
    <(zcat ../ref/hg38/utr5_hg38.bed.gz) \
    <(zcat ../ref/hg38/utr3_hg38.bed.gz) \
    > ../ref/hg38/utr_hg38.bed

! sort -k1,1 -k2,2n ../ref/utr_hg38.bed | gzip > ../ref/hg38/utr_hg38_sorted.bed.gz

## Natural splice sites from GENCODE Release 47 (GRCh38.p14)

### Prepare a bed file containing the exonic region

In [ ]:
# Download the gtf file from release 47 from GENCODE. This contains gene annotations.
! wget ref/hg38/https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.chr_patch_hapl_scaff.annotation.gtf.gz

# Convert the gtf file to a bed file and extract the exonic regions.
! gtfToGenePred -genePredExt ref/hg38/gencode.v47.annotation.gtf /dev/stdout \
    | gzip > ref/hg38/gencode.v47.annotation.genepred.txt.gz
! zcat ref/hg38/gencode.v47.annotation.genepred.txt.gz | awk '$6!=$7 && $8>1 && $3~/+/' | genePredToBed stdin ref/hg38/gencode.v47.annotation.bed
! bed12ToBed6 -i ref/hg38/gencode.v47.annotation.bed \
    | awk 'seen[$4]++' | sort -k1,1 -k2,2nr | awk 'seen[$4]++' \
    | sort -k4,4 | join -1 4 -2 1 -t$'\t' -o 1.1,1.2,1.3,2.1,2.2,1.6 \
    - <(zcat ref/hg38/gencode.v47.annotation.genepred.txt.gz | cut -f1,12 | sort -k1,1) \
    | sort -k5,5 | awk '!seen[$1, $2, $3, $5, $6]++' > ref/hg38/exons.bed

### Extract the coordinates of spliced sites (true labels)

#### Natural major splice sites

In [ ]:
# Identify the coordinates of the spliced sites (boundaries of the exons).
! awk 'BEGIN{OFS="\t"} {print $1,$3,$3+2,"donor |" $5,".",$6}' ref/hg38/exons.bed > ref/hg38/ss_hg38.bed
! awk 'BEGIN{OFS="\t"} {print $1,$2-2,$2,"acceptor |" $5,".",$6}' ref/hg38/exons.bed >> ref/hg38/ss_hg38.bed

# Extract the coordinates of the natural major spliced sites 
! bedtools getfasta -bed ref/hg38/ss_hg38.bed -fi ../../../dotseq/ref/hg38/hg38.fa -tab -name -s \
    | awk '(toupper($NF)~/GT/ && /donor/) || (toupper($NF)~/AG/ && /acceptor/)' | sed 's/|/\t/;s/::/\t/' \
    | awk 'BEGIN{OFS="\t"} {print $3,$1,$2}' | sed 's/:/\t/;s/-/\t/;s/(/\t/;s/)//' \
    | awk 'BEGIN{OFS="\t"} !seen[$1,$2,$3,$4]++ {print $1,$2,$3,$5,$6,$4}' > ref/hg38/ss_nat_major_hg38.bed

#### Natural minor and non-canonical splice sites

In [ ]:
# This command extracts both canonical and non-canonical dinucleotides
# I used this so that I can get the actual non-canonical dinucletides. The previous code from Lim was for canonical and hard coded to be GT/AG
# ! bedtools getfasta -bed ../ref/hg38/ss_hg38.bed -fi ../../../../dotseq/ref/hg38/hg38.fa -tab -name -s | awk 'BEGIN{OFS="\t"} {
#     seq=$2;

#     split($1, a, "\\|");

#     gene = a[2];
#     label = a[1];

#     split(gene, b, "::");
#     id = b[1];
#     loc = b[2];

#     match(loc, /(chr[^:]+):([0-9]+)-([0-9]+)\(([+-])\)/, m);

#     chr = m[1];
#     start = m[2];
#     end = m[3];
#     strand = m[4];

#     sub(/^[A-Z]{2}/, seq, label);

#     print chr, start, end, label, id, strand;
#     }' > ref/all_ss.bed

# Extract the minor and non-canonical dinucleotides only
df = pd.read_csv('ref/all_ss.bed', sep='\t', header=None)
df = df.sort_values([0,1,2,4]).reset_index(drop=True)
df[3] = df[3].str.replace(r'^[A-Za-z]{2}', lambda m: m.group(0).upper(), regex=True)
df_nc = df[(df[3]!='GT(donor)') & (df[3]!='AG(acceptor)') & (df[3]!='GC(donor)') & (df[3]!='AT(donor)') & (df[3]!='AC(acceptor)')].reset_index(drop=True)
df_nc.to_csv('ref/nc_ss.bed',index=None, sep='\t',header=None)
df_minor = df[(df[3]=='GC(donor)') | (df[3]=='AT(donor)') | (df[3]=='AC(acceptor)')].reset_index(drop=True)
df_minor.to_csv('ref/minor_ss.bed',index=None, sep='\t',header=None)

### Extract the coordinates of the non-spliced sites (decoys)
#### this is only done for natural major splice sites

In [ ]:
# This function converts fasta file to a pandas dataframe and is obtained from https://github.com/lcscs12345/HBV_splicing_paper_2025/blob/main/scripts/common.py
def fasta_to_dataframe(seq):
    fasta_df = pd.read_csv(seq, sep='>', lineterminator='>', header=None)
    df = fasta_df[0].str.split('\n', n=1, expand=True)
    df[1] = df[1].replace('\n','', regex=True)
    df = df[df[1] != '']
    df = df.dropna()
    df.columns = ['tid','seq']
    return df

#### Natural major splice sites

In [ ]:
# This function extracts the coordinates of the non-spliced sites and is obtained from https://github.com/lcscs12345/HBV_splicing_paper_2025/blob/main/scripts/splicebert_helpers.py
def nonss(fasta, chrom, start, end):
    fd = fasta[fasta.tid==chrom].copy()
    sequence = fd.seq.iloc[0].upper()
    
    positions = []
    for i in range(int(start), int(end)):
        if sequence[i:i+2] == "GT":
            positions.append([chrom, i, i+2, "GT(non-donor)", "none", "+"])
        elif sequence[i:i+2] == "AG":
            positions.append([chrom, i, i+2, "AG(non-acceptor)", "none", "+"])
        elif sequence[i:i+2] == "AC":
            positions.append([chrom, i, i+2, "GT(non-donor)", "none", "-"])
        elif sequence[i:i+2] == "CT":
            positions.append([chrom, i, i+2, "AG(non-acceptor)", "none", "-"])
            
    return pd.DataFrame(positions)

fa = fasta_to_dataframe("../../../dotseq/ref/hg38/hg38.fa")
exons = pd.read_csv("ref/hg38/exons_hg38.bed", sep="\t", header=None).sample(frac=0.05, random_state=42)
nonssbed = pd.concat(exons.apply(lambda x: nonss(fa, x[0], x[1], x[2]), axis=1).tolist())
# nonssbed.to_csv("ref/hg38/nonss_hg38.bed", sep="\t", index=None, header=None)

# Append the decoys to the true splice sites
! cat ref/hg38/ss_nat_major_hg38.bed ref/hg38/nonss_hg38.bed | awk '!seen[$1,$2,$3,$6]++' | gzip > ref/hg38/splice_sites_hg38.bed.gz

In [ ]:
# This is a modified function obtained from https://github.com/lcscs12345/HBV_splicing_paper_2025/blob/main/scripts/splicebert_helpers.py
# Find the minor dinucleotide that are not at the splice junction in GENCODE
def nonss(fasta, chrom, start, end):
    fd = fasta[fasta.tid==chrom].copy()
    sequence = fd.seq.iloc[0].upper()
    
    positions = []
    for i in range(int(start), int(end)):
        if sequence[i:i+2] == "GC":
            positions.append([chrom, i, i+2, "GC(non-donor)", "none", "+"])
        elif sequence[i:i+2] == "AT":
            positions.append([chrom, i, i+2, "AT(non-donor)", "none", "+"])
        elif sequence[i:i+2] == "AC":
            positions.append([chrom, i, i+2, "AC(non-acceptor)", "none", "+"])
        elif sequence[i:i+2] == "GC":
            positions.append([chrom, i, i+2, "GC(non-donor)", "none", "-"])
        elif sequence[i:i+2] == "AT":
            positions.append([chrom, i, i+2, "AT(non-acceptor)", "none", "-"])
        elif sequence[i:i+2] == "GT":
            positions.append([chrom, i, i+2, "GT(non-acceptor)", "none", "-"])
            
    return pd.DataFrame(positions)

exons = pd.read_csv("/home/chiga034/projects_limgroup/gab/2026/rna_llm/ref/hg38/exons_hg38.bed", sep="\t", header=None).sample(frac=0.05, random_state=42)
fa = fasta_to_dataframe("/home/chiga034/projects_limgroup/dotseq/ref/hg38/hg38.fa")
nonssbed = pd.concat(exons.apply(lambda x: nonss(fa, x[0], x[1], x[2]), axis=1).tolist())
# nonssbed.to_csv("ref/minor_nonss_hg38.bed", sep="\t", index=None, header=None)

# combine the canonical_minor spliced and canonical_minor non-splice sites, drop duplicates, sort and zip the file
! cat ref/minor_ss.bed ref/minor_nonss_hg38.bed | awk '!seen[$1,$2,$3,$6]++' | sort -k1,1 -k2,2n | gzip > ref/splice_sites_minor_hg38.bed.gz
! zcat ref/splice_sites_minor_hg38.bed.gz | sort -k1,1 -k2,2n | gzip > ref/splice_sites_minor_hg38_sorted.bed.gz

In [ ]:
# This is a modified function obtained from https://github.com/lcscs12345/HBV_splicing_paper_2025/blob/main/scripts/splicebert_helpers.py
# Find the non-canonical dinucleotide that are not at the splice junction in GENCODE    
def nonss(fasta, chrom, start, end, valid_set):
    fd = fasta[fasta.tid == chrom]
    sequence = fd.seq.iloc[0].upper()

    positions = []

    for i in range(int(start), int(end) - 1):
        dinuc = sequence[i:i+2]

        for label in valid_set:
            if label.startswith(dinuc):

                base = label[:2]

                if "donor" in label:
                    new_label = f"{base}(non-donor)"
                    strand = "+"

                else:
                    new_label = f"{base}(non-acceptor)"
                    strand = "-"

                positions.append([
                    chrom,
                    i,
                    i+2,
                    new_label,
                    "none",
                    strand
                ])

    return pd.DataFrame(positions)

exons = pd.read_csv("/home/chiga034/projects_limgroup/gab/2026/rna_llm/ref/hg38/exons_hg38.bed", sep="\t", header=None).sample(frac=0.05, random_state=42)
fa = fasta_to_dataframe("/home/chiga034/projects_limgroup/dotseq/ref/hg38/hg38.fa")

valid_set = df_nc[3].unique()

nonssbed = pd.concat(
    exons.apply(
        lambda x: nonss(fa, x[0], x[1], x[2], valid_set),
        axis=1
    ).tolist()
)
nonssbed = nonssbed.sort_values([0,1,2])
# nonssbed[3] = (nonssbed[3].str.replace(r'\(donor\)', '(non-donor)', regex=True).str.replace(r'\(acceptor\)', '(non-acceptor)', regex=True)) # forgot to add the code in the function definition to change it to 'non-donor' and 'non-acceptor'
# nonssbed.to_csv("ref/non_canonical_nonss_hg38.bed", sep="\t", index=None, header=None)

# combine the non-canonical spliced and non-canonical non-splice sites, drop duplicates, sort and zip the file
! cat ref/nc_ss.bed ref/non_canonical_nonss_hg38.bed | awk '!seen[$1,$2,$3,$6]++' | sort -k1,1 -k2,2n | gzip > ref/splice_sites_non_canonical_hg38.bed.gz
! zcat ref/splice_sites_non_canonical_hg38.bed.gz | sort -k1,1 -k2,2n | gzip > ref/splice_sites_non_canonical_hg38_sorted.bed.gz

### Map natural major splice sites to their respective regions

In [ ]:
# Natural major splice sites
! zcat ref/hg38/splice_sites_hg38.bed.gz | sort -k1,1 -k2,2n | gzip > ref/hg38/splice_sites_hg38_sorted.bed.gz
! bedtools intersect \
  -a <(zcat ref/hg38/splice_sites_hg38_sorted.bed.gz) \
  -b <(zcat ref/hg38/utr_hg38_sorted.bed.gz) \
  -loj -wa -wb \
  | gzip > ref/hg38/splice_sites_final_hg38.bed.gz

df = pd.read_csv('ref/hg38/splice_sites_final_hg38.bed.gz', sep='\t', header=None)
df[9] = df[9].replace({'.': 'CDS'})
df[3] = df[3].str.rstrip(')').str.split(')').str[0] + "_" + df[9] + ")" # cannot use '|' because there is a line in fetch_embeddings.py that uses '|' to split the name column
df = df[[0,1,2,3,4,5]]
df.to_csv("ref/hg38/splice_sites_sb_input_hg38.bed", sep="\t", header=None, index=None)
! gzip ref/hg38/splice_sites_sb_input_hg38.bed

# Natural minor splite sites
! cat ref/minor_ss.bed | awk '!seen[$1,$2,$3,$6]++' | sort -k1,1 -k2,2n | gzip > ref/minor_ss_sorted.bed.gz
! bedtools intersect \
  -a <(zcat ref/minor_ss_sorted.bed.gz) \
  -b <(zcat /home/chiga034/projects_limgroup/gab/2026/rna_llm/ref/hg38/utr_hg38_sorted.bed.gz) \
  -loj -wa -wb \
  | gzip > ref/minor_ss_final.bed.gz

df = pd.read_csv('ref/minor_ss_final.bed.gz', sep='\t', header=None)
df[9] = df[9].replace({'.': 'CDS'})
df[3] = df[3].str.rstrip(')').str.split(')').str[0] + "_" + df[9] + ")" #cannot use '|' because there is a line in fetch_embeddings.py that uses '|' to split the name column
df = df[[0,1,2,3,4,5]]
# df.to_csv("ref/uncombined_minor_sb_input_hg38.bed", sep="\t", header=None, index=None)
# ! gzip ref/uncombined_minor_sb_input_hg38.bed

# Natural non-canonical splite sites
! cat ref/nc_ss.bed | awk '!seen[$1,$2,$3, $4, $5, $6]++' | sort -k1,1 -k2,2n | gzip > ref/nc_ss_sorted.bed.gz
! bedtools intersect \
  -a <(zcat ref/nc_ss_sorted.bed.gz) \
  -b <(zcat /home/chiga034/projects_limgroup/gab/2026/rna_llm/ref/hg38/utr_hg38_sorted.bed.gz) \
  -loj -wa -wb \
  | gzip > ref/nc_ss_final.bed.gz

df = pd.read_csv('ref/nc_ss_final.bed.gz', sep='\t', header=None)
df[9] = df[9].replace({'.': 'CDS'})
df[3] = df[3].str.rstrip(')').str.split(')').str[0] + "_" + df[9] + ")" #cannot use '|' because there is a line in fetch_embeddings.py that uses '|' to split the name column
df = df[[0,1,2,3,4,5]]
# df.to_csv("ref/uncombined_non_canonical_sb_input_hg38.bed", sep="\t", header=None, index=None)
# ! gzip ref/uncombined_non_canonical_sb_input_hg38.bed

## Cryptic splice sites from DBASS

### Download dataset from DBASS and extract the coordinates of the cryptic splice sites

In [ ]:
# Cryptic donor splice sites
df5 = pd.read_csv('data/dbass/dbass5-splice-sites.csv', sep=',')

def get_dinuc_after_slash(seq):
    seq = str(seq).lower()
    if '/' not in seq:
        return None

    pos = seq.index('/')
    dinuc = seq[pos+1:pos+3]

    # keep only valid A/C/G/T dinucleotides
    if len(dinuc) == 2 and dinuc.isalpha():
        return dinuc
    return None


df5['din_seq'] = df5['NucleotideSequence'].apply(get_dinuc_after_slash)

df5['ss_type'] = df5['din_seq'].apply(
    lambda x: f"{x.upper()}(donor)_cryptic" if pd.notna(x) else None
)

# Cryptic acceptor splice sites
df3 = pd.read_csv('data/dbass/dbass3-splice-sites.csv', sep=',')
def get_dinuc_before_slash(seq):
    seq = str(seq).lower()
    if '/' not in seq:
        return None

    pos = seq.index('/')
    dinuc = seq[pos-2:pos]

    # keep only valid A/C/G/T dinucleotides
    if len(dinuc) == 2 and dinuc.isalpha():
        return dinuc
    return None


df3['din_seq'] = df3['NucleotideSequence'].apply(get_dinuc_before_slash)

df3['ss_type'] = df3['din_seq'].apply(
    lambda x: f"{x.upper()}(acceptor)_cryptic" if pd.notna(x) else None
)

df = pd.concat([df3,df5]).reset_index(drop=True)
df = df.drop(columns=['Id'])
df = df.dropna(subset=['ss_type'])

In [ ]:
# Prepare a bed file 
bed = df[['AberrantSpliceSiteCoordinates', 'ss_type','EnsemblReference']]
bed['AberrantSpliceSiteCoordinates'] = bed['AberrantSpliceSiteCoordinates'].str.split(',')
bed = bed.explode('AberrantSpliceSiteCoordinates')
bed['AberrantSpliceSiteCoordinates'] = (bed['AberrantSpliceSiteCoordinates'].str.strip())
bed['chr'] = bed.AberrantSpliceSiteCoordinates.str.split(':').str[0]
bed['start'] = bed.AberrantSpliceSiteCoordinates.str.split('/').str[0].str.split(':').str[1]
bed['end'] = bed.AberrantSpliceSiteCoordinates.str.split('/').str[1]
bed = bed[['chr','start','end','ss_type','EnsemblReference','AberrantSpliceSiteCoordinates']]
bed['AberrantSpliceSiteCoordinates'] = 'none'

bed = bed.dropna(subset=['EnsemblReference']) # empty gid

bed = bed[
    bed['start'].astype(str).str.fullmatch(r'\d+') &
    bed['end'].astype(str).str.fullmatch(r'\d+')
].copy()

bed['start'] = bed['start'].astype(int)
bed['end'] = bed['end'].astype(int)
def fix_dbass_coords(row):

    s = row['start']

    if 'donor' in row['ss_type']:
        row['start'] = s
        row['end'] = s + 2

    elif 'acceptor' in row['ss_type']:
        row['start'] = s - 2
        row['end'] = s

    return row
bed = bed.apply(fix_dbass_coords, axis=1)
# bed.to_csv('cryptic_dbass.bed', sep='\t', index=None, header=None)

### Map cryptic splice sites to their respective regions

In [ ]:
# sort bed file prior to intersecting
! cat cryptic_dbass.bed | sort -k1,1 -k2,2n | gzip > cryptic_dbass_sorted.bed.gz
! bedtools intersect \
  -a <(zcat cryptic_dbass_sorted.bed.gz) \
  -b <(zcat /home/chiga034/projects_limgroup/gab/2026/rna_llm/ref/hg38/utr_hg38_sorted.bed.gz) \
  -loj -wa -wb \
  | gzip > cryptic_dbass_annotated_with_region.bed.gz

df = pd.read_csv('cryptic_dbass_annotated_with_region.bed.gz', sep='\t', header=None)
df[9] = df[9].replace({'.': 'CDS'})
df[3] = df.apply(lambda row: row[3].replace('(', f'({row[9]}_'),axis=1) #cannot use '|' because there is a line in fetch_embeddings.py that uses '|' to split the name column
df = df[[0,1,2,3,4,5]]
# df.to_csv("cryptic_dbass_sb_input.bed", sep="\t", header=None, index=None) # to extract embeddings
# ! gzip cryptic_dbass_sb_input.bed